# Lab02BIA — Tarefa 03  
## Análise exploratória e regressão linear para qualidade de vinhos

Este notebook foi organizado a partir do código original.  
Foram adicionados comentários em português e algumas observações didáticas para facilitar o estudo.

> **Importante:** ajuste os caminhos dos arquivos `winequality-red.csv` e `winequality-white.csv` para a pasta onde eles estão salvos no seu computador.


## 1. Importação das bibliotecas


In [ ]:
# cross_val_score: calcula a métrica do modelo por validação cruzada.
# StratifiedKFold: separa os dados em grupos mantendo a proporção das classes de qualidade.
# LinearRegression: cria o modelo de regressão linear.
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LinearRegression

# pandas: biblioteca para leitura, organização e análise de tabelas de dados.
import pandas as pd


## 2. Leitura das bases de dados


In [ ]:
# Lê a base de vinhos tintos.
# O parâmetro sep=';' informa que as colunas do arquivo CSV são separadas por ponto e vírgula.
# Substitua o caminho abaixo pelo local correto no seu computador.
df_rose = pd.read_csv(
    r'C:/Users/abalb/Desktop/winequality-red.csv',
    sep=';'
)

# Lê a base de vinhos brancos.
df_branco = pd.read_csv(
    r'C:/Users/abalb/Desktop/winequality-white.csv',
    sep=';'
)


## 3. Inspeção inicial das tabelas


In [ ]:
# Exibe o número de linhas, colunas, tipos de dados e valores não nulos
# da base de vinhos tintos.
df_rose.info()


In [ ]:
# Exibe as mesmas informações para a base de vinhos brancos.
df_branco.info()


In [ ]:
# Retorna uma tupla no formato (número de linhas, número de colunas).
df_branco.shape


## 4. Criação da variável que identifica o tipo de vinho


In [ ]:
# Cria a coluna 'wine_type' para identificar a origem de cada observação.
# Convenção utilizada:
# 1 = vinho tinto
# 0 = vinho branco
df_rose["wine_type"] = 1
df_branco["wine_type"] = 0


In [ ]:
# Mostra as cinco primeiras linhas da base de vinhos tintos.
df_rose.head()


## 5. União das duas bases


In [ ]:
# Concatena as linhas das duas tabelas em uma única base.
# Como as colunas possuem a mesma estrutura, cada DataFrame é empilhado verticalmente.
df_bruto = pd.concat([df_rose, df_branco], ignore_index=True)

# Exibe o DataFrame combinado.
df_bruto


In [ ]:
# iloc seleciona linhas pela posição numérica.
# Aqui são exibidas as duas primeiras e as duas últimas linhas.
df_bruto.iloc[[0, 1, -2, -1]]


In [ ]:
# Outro exemplo de seleção por posição:
# duas primeiras linhas e a terceira/segunda linhas a partir do final.
df_bruto.iloc[[0, 1, -2, -3]]


In [ ]:
# Verifica novamente estrutura, tipos de dados e valores não nulos
# após unir as duas bases.
df_bruto.info()


In [ ]:
# Mostra o índice das linhas do DataFrame.
df_bruto.index


## 6. Estatística descritiva e verificação de dados faltantes


In [ ]:
# Calcula estatísticas descritivas apenas para as variáveis selecionadas:
# contagem, média, desvio-padrão, mínimo, quartis e máximo.
df_bruto[
    ["fixed acidity", "residual sugar", "density", "alcohol", "quality"]
].describe()


In [ ]:
# Calcula as estatísticas descritivas para todas as colunas numéricas.
df_bruto.describe()


In [ ]:
# Conta a quantidade de valores ausentes (NaN) em cada coluna.
df_bruto.isnull().sum()


In [ ]:
# Conta quantas amostras existem em cada nível da variável alvo 'quality'.
# Isso ajuda a observar se a distribuição das notas é equilibrada ou desbalanceada.
df_bruto["quality"].value_counts().sort_index()


## 7. Visualização das distribuições


In [ ]:
import matplotlib.pyplot as plt

# Cria histogramas para todas as variáveis numéricas.
# bins define a quantidade de faixas do histograma.
# figsize define o tamanho da figura em polegadas.
df_bruto.hist(bins=100, figsize=(10, 12))
plt.tight_layout()
plt.show()


In [ ]:
# Filtra apenas os vinhos com qualidade maior que 7
# e calcula suas estatísticas descritivas.
df_bruto[df_bruto["quality"] > 7].describe()


In [ ]:
# Seleciona somente a coluna 'alcohol' dos vinhos de alta qualidade.
# A condição quality >= 7 cria uma máscara booleana para filtrar as linhas.
vinho_alta_qualidade = df_bruto.loc[df_bruto["quality"] >= 7, "alcohol"]
vinho_alta_qualidade


In [ ]:
# Seleciona somente a coluna 'alcohol' dos vinhos de baixa qualidade.
# Neste exemplo, foram definidos como baixa qualidade os vinhos com nota menor que 6.
vinho_baixa_qualidade = df_bruto.loc[df_bruto["quality"] < 6, "alcohol"]
vinho_baixa_qualidade


In [ ]:
# Compara graficamente a distribuição de álcool entre dois grupos de qualidade.
plt.hist(vinho_alta_qualidade, bins=10, alpha=0.7, label="Alta qualidade (>= 7)")
plt.hist(vinho_baixa_qualidade, bins=10, alpha=0.7, label="Baixa qualidade (< 6)")
plt.xlabel("Teor alcoólico")
plt.ylabel("Frequência")
plt.title("Distribuição do teor alcoólico por faixa de qualidade")
plt.legend(loc="upper right")
plt.show()


## 8. Correlação entre as variáveis


In [ ]:
# Calcula a matriz de correlação linear de Pearson.
# round(2) arredonda os valores para duas casas decimais.
matriz_corr = df_bruto.corr(numeric_only=True).round(2)
matriz_corr


In [ ]:
# Converte todos os valores da matriz em valores absolutos.
# Assim, uma correlação de -0,80 e +0,80 passam a ter a mesma intensidade: 0,80.
matriz_corr_abs = matriz_corr.abs()

# Destaca visualmente correlações com módulo maior que 0,40.
matriz_corr_abs[matriz_corr_abs > 0.40].style.background_gradient(cmap="Reds")


In [ ]:
# Exibe a correlação entre dióxido de enxofre total e dióxido de enxofre livre
# com todas as demais variáveis.
matriz_corr[["total sulfur dioxide", "free sulfur dioxide"]]


In [ ]:
# Remove 'total sulfur dioxide' da base usada no modelo.
# A decisão pode ser justificada pela correlação com 'free sulfur dioxide',
# reduzindo redundância entre atributos.
df = df_bruto.drop("total sulfur dioxide", axis=1)

# Mostra uma linha para confirmar as colunas restantes.
df.head(1)


In [ ]:
# Ordena as correlações com a variável alvo 'quality' da maior para a menor.
# Atenção: correlação indica associação linear; não prova causalidade.
matriz_corr["quality"].sort_values(ascending=False)


## 9. Separação entre atributos de entrada e variável alvo

- `X`: variáveis de entrada usadas para estimar a qualidade.
- `y`: variável alvo, isto é, a nota de qualidade do vinho.
- A divisão 80/20 reserva 20% das amostras para avaliação final do modelo.


In [ ]:
from sklearn.model_selection import train_test_split

# Remove a coluna 'quality' de X, pois ela é o resultado que desejamos prever.
X = df.drop(["quality"], axis=1)

# Define a variável alvo.
y = df["quality"]

# Separa dados de treino e teste.
# random_state=33 fixa a aleatoriedade e permite reproduzir a mesma divisão.
X_trein, X_teste, y_trein, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=33
)


## 10. Normalização e padronização dos atributos


In [ ]:
# MinMaxScaler transforma cada variável aproximadamente para o intervalo [0, 1].
# StandardScaler centraliza a média em 0 e ajusta o desvio-padrão para 1.
from sklearn.preprocessing import MinMaxScaler, StandardScaler

min_max_normalizar = MinMaxScaler()
std_normalizar = StandardScaler()

# fit_transform aprende os parâmetros usando APENAS o conjunto de treino
# e devolve os dados transformados. Isso evita vazamento de informação.
minmax_normalizado_X_trein = min_max_normalizar.fit_transform(X_trein)
std_normalizado_X_trein = std_normalizar.fit_transform(X_trein)

# Mantém também o conjunto de teste transformado com os parâmetros aprendidos no treino.
minmax_normalizado_X_teste = min_max_normalizar.transform(X_teste)
std_normalizado_X_teste = std_normalizar.transform(X_teste)


In [ ]:
# Converte o resultado normalizado em DataFrame para preservar os nomes das colunas.
df_minmax_normalizado_X = pd.DataFrame(
    minmax_normalizado_X_trein,
    columns=X.columns,
    index=X_trein.index
)
df_minmax_normalizado_X.head()


In [ ]:
# Faz o mesmo para os dados padronizados.
df_std_normalizado_X = pd.DataFrame(
    std_normalizado_X_trein,
    columns=X.columns,
    index=X_trein.index
)
df_std_normalizado_X.head()


## 11. Validação cruzada da regressão linear

A métrica escolhida é o **erro absoluto mediano**, retornado como valor negativo pelo scikit-learn quando usado em `cross_val_score`.  
Por isso, multiplicamos o resultado por `-1` para apresentar o erro com sinal positivo.

> Como `quality` possui valores discretos, `StratifiedKFold` ajuda a preservar aproximadamente a proporção das notas em cada divisão.


In [ ]:
# Cria 3 divisões estratificadas para validação cruzada.
kfold = StratifiedKFold(n_splits=3, random_state=33, shuffle=True)

# Avalia a regressão linear com os atributos normalizados.
# O scikit-learn retorna métricas de erro com sinal negativo por convenção.
lr_cve_minmax = cross_val_score(
    LinearRegression(),
    df_minmax_normalizado_X,
    y_trein,
    cv=kfold,
    scoring="neg_median_absolute_error"
)

print(f"Erro absoluto mediano médio — dados normalizados: {-lr_cve_minmax.mean():.3f}")


In [ ]:
# Avalia a regressão linear com os atributos padronizados.
lr_cve_std = cross_val_score(
    LinearRegression(),
    df_std_normalizado_X,
    y_trein,
    cv=kfold,
    scoring="neg_median_absolute_error"
)

print(f"Erro absoluto mediano médio — dados padronizados: {-lr_cve_std.mean():.3f}")


## 12. Treinamento final e avaliação no conjunto de teste

A regressão linear será treinada diretamente com `X_trein`.  
Para este modelo específico, escalar os atributos não muda, em geral, as previsões finais, pois os coeficientes se ajustam à escala. Porém, normalização e padronização continuam importantes para muitos outros modelos.


In [ ]:
# Ajusta (treina) o modelo de regressão linear usando o conjunto de treino.
lr = LinearRegression().fit(X_trein, y_trein)

# Mostra o objeto do modelo criado.
print(lr)


In [ ]:
# Gera as previsões de qualidade para as amostras que não participaram do treinamento.
y_pred = lr.predict(X_teste)

# Exibe as previsões.
y_pred


In [ ]:
# Exibe apenas uma parte das previsões para facilitar a inspeção.
print(y_pred[5:20])


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# MAE: erro absoluto médio entre valores reais e previstos.
mae_lr = mean_absolute_error(y_teste, y_pred)

# MSE: média dos erros ao quadrado; penaliza mais fortemente erros grandes.
mse_lr = mean_squared_error(y_teste, y_pred)

# R²: proporção aproximada da variação de y explicada pelo modelo.
r2_wine = r2_score(y_teste, y_pred)

print(f"MAE da Regressão Linear: {mae_lr:.3f}")
print(f"MSE da Regressão Linear: {mse_lr:.3f}")
print(f"R² da Regressão Linear: {r2_wine:.3f}")


## Conclusão

Este notebook executa uma sequência básica de análise de dados:

1. Leitura e união de duas bases de vinhos;
2. Inspeção da estrutura, estatísticas e valores ausentes;
3. Visualização de distribuições e análise de correlação;
4. Preparação de atributos e variável alvo;
5. Normalização/padronização;
6. Treinamento, validação cruzada e avaliação de uma regressão linear.

Para executar no VS Code, abra o arquivo `.ipynb`, selecione o kernel da sua `.venv` e ajuste os caminhos dos arquivos CSV.
